# Predicting module outcome from a graph embedding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jose-alvarado-guzman/oulad/blob/main/notebooks/aga_outcome_prediction.ipynb)

Embeds each student with **FastRP** over their engagement and demographic neighbourhood, then
trains a **node classification pipeline** to predict whether they pass the module.

FastRP embeds *topology* — who a student is connected to — rather than a sequence. A student
is characterised by the materials they touched and the age band, region, prior education and
deprivation band they belong to, all of which are already nodes in this graph. That is the
kind of structure a random projection is good at compressing.

The pipeline follows the GDS machine-learning docs:
<https://neo4j.com/docs/graph-data-science/current/machine-learning/machine-learning/>

```
fastRP node property  ->  select features  ->  split  ->  candidate models  ->  train
```

Two deliberate choices, both about honesty rather than accuracy:

- **Assessment relationships are excluded.** `WAS_ASSESSED_IN` carries scores, and scores
  determine `finalResult` almost by definition. Predicting an outcome from the marks that
  produced it is leakage dressed up as a model. Only engagement and demographics go in, which
  is also the only version of the question that could support an early intervention.
- **A trivial baseline is trained alongside it.** Step 10 predicts pass/fail from
  `sum(sumClick)` and nothing else. A 128-dimensional embedding that cannot beat one `sum()`
  has not earned its place, and on this dataset that is a real possibility rather than a
  rhetorical one.

## Before you start

The usual secrets: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `AURA_CLIENT_ID`,
`AURA_CLIENT_SECRET`, `AURA_PROJECT_ID`, with *Notebook access* on.

> **This notebook does not write to your database.** The class label is computed inside the
> projection query, so nothing is stored on any node. The only thing to release is the
> session, which step 12 deletes.

## 1. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    os.path.join(REPO_DIR, 'requirements-aga.txt')], check=True)
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory, AuraAPICredentials, DbmsConnectionInfo, GdsSessions)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS, ETL_SECRETS, MissingCredentialsError, aura_instance_id, load_credentials)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'], os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID']))

## 4. Scope and class balance

One module. `passed` is 1 for Pass or Distinction and 0 for Fail or Withdrawn — a binary
target, which is both easier to learn and closer to the question a support team would ask.
Change `PASS_RESULTS` for a different cut, or drop the binarisation entirely and train on all
four classes.

Class balance matters for reading the metrics later: if 60% of students pass, a model that
predicts "pass" for everyone scores 60% accuracy while being useless. That is why F1 macro is
the metric the pipeline selects on.

In [ ]:
MODULE = 'GGG'
PASS_RESULTS = ['Pass', 'Distinction']

BALANCE_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
RETURN finalResult, count(*) AS students ORDER BY students DESC
'''
with driver.session(database=NEO4J_DATABASE) as session:
    balance = pd.DataFrame(session.run(BALANCE_QUERY, module=MODULE).data())

balance['class'] = balance['finalResult'].isin(PASS_RESULTS).map({True: 'passed', False: 'not'})
print(f'module {MODULE} outcomes')
print(balance.to_string(index=False))
summary = balance.groupby('class')['students'].sum()
majority = summary.max() / summary.sum() * 100
print(f'\nbinary split: {summary.to_dict()}')
print(f'a model that always guesses the majority class scores {majority:.1f}% accuracy')

## 5. Size and open the session

In [ ]:
SIZE_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s
MATCH (s)-[r:REVIEWED_MATERIAL|IN_AGE_GROUP|LIVE_IN_REGION|HAS_HIGHER_EDUCATION
           |IN_DEPRIVATION_GROUP]->(t)
RETURN count(DISTINCT s) AS students, count(DISTINCT t) AS neighbours, count(r) AS edges
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sized = session.run(SIZE_QUERY, module=MODULE).single()

node_count = sized['students'] + sized['neighbours']
relationship_count = sized['edges']
print(f"{sized['students']:,} students + {sized['neighbours']:,} neighbour nodes "
      f'= {node_count:,} nodes, {relationship_count:,} relationships')

memory = sessions.estimate(
    node_count=node_count, relationship_count=relationship_count,
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING,
                          AlgorithmCategory.MACHINE_LEARNING],
    node_label_count=6, node_property_count=2)
print('estimated memory:', memory)

SESSION_NAME = f"oulad-predict-{os.environ['AURA_CLIENT_ID'][:8]}"
gds = sessions.get_or_create(
    session_name=SESSION_NAME, memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID, username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD, database=NEO4J_DATABASE),
    ttl=timedelta(hours=2))
print('session ready:', SESSION_NAME)

## 6. Project the neighbourhood, label included

The label is computed **in the projection query** and attached as a node property, so the class
never has to be written to your database. It exists only inside the session.

The relationship list is the graph half of the feature space: materials touched, plus the four
demographic dimensions. `WAS_ASSESSED_IN` is absent on purpose — see the note at the top.

`logClicks` is the other half, and it is here for a reason discovered the hard way. **FastRP
normalises its embeddings, which suppresses degree** — so an embedding built from this graph
knows *which* materials a student touched but barely encodes *how many*. Trained on the
embedding alone, the pipeline collapsed to predicting "pass" for all 2,525 students: F1 macro
0.372, accuracy 0.592, against a majority-class rate of 0.598. Volume has to be handed to the
model explicitly. It is logged because raw click totals run from 0 to 10,768 and that skew is
poison for logistic regression.

`REVIEWED_MATERIAL` is left unaggregated, so a student who returned to a page many times has
many parallel edges to it — implicit frequency weighting for the random projection.

In [ ]:
GRAPH_NAME = 'oulad-prediction'

PROJECTION_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
WITH s, CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END AS passed
OPTIONAL MATCH (s)-[rm:REVIEWED_MATERIAL]->(:EducationalMaterial)
              <-[:HAS_MATERIAL]-(cm:Course)
WHERE cm.codeModule = $module
WITH s, passed, coalesce(sum(rm.sumClick), 0) AS clicks
MATCH (s)-[r:REVIEWED_MATERIAL|IN_AGE_GROUP|LIVE_IN_REGION|HAS_HIGHER_EDUCATION
           |IN_DEPRIVATION_GROUP]->(t)
RETURN gds.graph.project.remote(s, t, {
    sourceNodeLabels: labels(s),
    targetNodeLabels: labels(t),
    sourceNodeProperties: s {
        passed: passed,
        logClicks: log(toFloat(clicks) + 1.0)
    },
    relationshipType: type(r)
})
'''

gds.graph.project.cypher(
    graph_name=GRAPH_NAME, query=PROJECTION_QUERY,
    query_parameters={'module': MODULE, 'passResults': PASS_RESULTS},
    overwrite=True)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes and {G.relationship_count():,} relationships')
print('node properties:', G.node_properties())

student_properties = set(G.node_properties().get('Student', []))
missing = {'passed', 'logClicks'} - student_properties
if missing:
    raise SystemExit(f'{missing} did not arrive on Student; training would be built on '
                     'nothing. Check sourceNodeProperties in PROJECTION_QUERY.')
print("\nlabel 'passed' and feature 'logClicks' are on the Student nodes")

## 7. A pipeline you can train twice

Wrapped in a function, because the notebook trains it **two ways**: with the embedding plus
volume, and with volume alone. Without that second run there is no way to tell whether the 128
dimensions are doing anything, and the honest answer on this data is that they are not — see
step 10.

`add_node_property` puts FastRP *inside* the pipeline, so the embedding is recomputed per fold
rather than once up front. Two candidate models are offered and the pipeline keeps whichever
scores better on the validation folds.

In [ ]:
EMBEDDING = 'embedding'

def train_variant(features, suffix):
    """Build a pipeline over `features`, train it, and return its test metrics."""
    pipeline_name = f'oulad-pipeline-{suffix}'
    model_name = f'oulad-model-{suffix}'
    for getter in (lambda: gds.model.get(model_name).drop(),
                   lambda: gds.pipeline.node_classification.get(pipeline_name).drop()):
        try:
            getter()
        except Exception:
            pass

    pipe, _ = gds.pipeline.node_classification.create(pipeline_name)
    pipe.add_node_property(
        'fastRP', mutate_property=EMBEDDING, embedding_dimension=128,
        iteration_weights=[0.0, 1.0, 1.0, 1.0], normalization_strength=-0.9,
        random_seed=42)
    pipe.select_features(features)
    pipe.configure_split(test_fraction=0.3, validation_folds=4)
    pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
    pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)

    model, _ = pipe.train(
        G, model_name=model_name, metrics=['F1_MACRO', 'ACCURACY'],
        target_property='passed', target_node_labels=['Student'], random_seed=42)
    scores = model.metrics() or {}
    return model, {m: v.get('test') for m, v in scores.items() if isinstance(v, dict)}

VARIANTS = {
    'embedding + volume': ([EMBEDDING, 'logClicks'], 'full'),
    'volume only':        (['logClicks'], 'ablation'),
}
print('variants to train:')
for label, (features, _) in VARIANTS.items():
    print(f'  {label:20s} features={features}')

## 8. The baseline, before any training

One feature, no model: predict pass if a student's clicks are above the median. This is the
bar, and it is computed straight from the database with no session involved.

In [ ]:
BASELINE_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
OPTIONAL MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module
RETURN s.id AS studentId,
       CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END AS passed,
       coalesce(sum(r.sumClick), 0) AS clicks
'''
with driver.session(database=NEO4J_DATABASE) as session:
    base = pd.DataFrame(session.run(
        BASELINE_QUERY, module=MODULE, passResults=PASS_RESULTS).data())

threshold = base['clicks'].median()
base['predicted'] = (base['clicks'] >= threshold).astype(int)

def macro_f1(truth, predicted):
    scores = []
    for label in (0, 1):
        tp = ((predicted == label) & (truth == label)).sum()
        fp = ((predicted == label) & (truth != label)).sum()
        fn = ((predicted != label) & (truth == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall)
                      if precision + recall else 0.0)
    return sum(scores) / len(scores)

BASELINE = {'F1_MACRO': macro_f1(base['passed'], base['predicted']),
            'ACCURACY': (base['predicted'] == base['passed']).mean()}
print(f"baseline: pass if clicks >= {threshold:,.0f} (the median), "
      f"over {len(base):,} students")
print(f"  F1 macro {BASELINE['F1_MACRO']:.4f}, accuracy {BASELINE['ACCURACY']:.4f}")

## 9. Train both variants

Two trainings, a few seconds each. The second exists only to answer "would one feature have
done just as well?".

In [ ]:
models, results = {}, {}
for label, (features, suffix) in VARIANTS.items():
    print(f'\ntraining: {label}', flush=True)
    model, scores = train_variant(features, suffix)
    models[label] = model
    results[label] = scores
    print(f'  {scores}')
    print(f'  winning candidate: '
          f"{(model.best_parameters() or {}).get('methodName', 'unknown')}")

## 10. What the embedding was worth

Read the gap between the two trained rows, not the gap to the baseline. The baseline is a
crude threshold, so any model beats it; the question is whether **128 embedding dimensions
beat one logged number**.

On module GGG they do not — the difference is a fraction of a point on a ~760-student test
split, which is noise. The predictive signal is engagement volume, and FastRP's normalisation
actively suppresses it, which is why the embedding *alone* collapses to predicting the
majority class for everyone.

That is a result about this dataset, not about graph embeddings. It would look different on a
graph where structure carries information that no single aggregate captures — recommendation,
fraud rings, citation communities. Here it does not.

In [ ]:
rows = [{'method': label, **scores} for label, scores in results.items()]
rows.append({'method': 'clicks >= median (no model)', **BASELINE})
comparison = pd.DataFrame(rows).set_index('method').round(4)
print(comparison.to_string())

full = results.get('embedding + volume', {})
ablated = results.get('volume only', {})
if full.get('ACCURACY') and ablated.get('ACCURACY'):
    gain = (full['ACCURACY'] - ablated['ACCURACY']) * 100
    print(f'\nembedding contributes {gain:+.2f} accuracy points over volume alone')
    print('that is within noise on this test split' if abs(gain) < 1.5
          else 'that is a real difference on this test split')

## 11. Predict, and look at what it gets wrong

Predictions on the whole graph, streamed rather than written, so the database stays untouched.
The interesting part is not the accuracy but *which* students the model misses — if the errors
concentrate on low-engagement students, the embedding is doing no better than the baseline
where it matters most.

In [ ]:
model = models['embedding + volume']
predictions = model.predict_stream(G, target_node_labels=['Student'])
print('columns:', list(predictions.columns))

ids = gds.graph.node_properties.stream(G, 'passed', node_labels=['Student'])
truth_column = [c for c in ids.columns if c != 'nodeId'][-1]
ids = ids.rename(columns={truth_column: 'actual'})

predicted_column = [c for c in predictions.columns
                    if c not in ('nodeId',) and 'probab' not in c.lower()][-1]
merged = predictions.rename(columns={predicted_column: 'predicted'})[['nodeId', 'predicted']]
merged = merged.merge(ids[['nodeId', 'actual']], on='nodeId')

confusion = pd.crosstab(merged['actual'], merged['predicted'],
                        rownames=['actual'], colnames=['predicted'])
print('\nconfusion matrix over all labelled students')
print(confusion.to_string())
print(f"\noverall agreement: {(merged['predicted'] == merged['actual']).mean():.4f}")

## 12. Clean up

Only session-side objects to release — the model, the pipeline and the projection all live in
the session, and the session itself is the billed part. **Nothing was written to your
database**, so there is nothing to undo.

In [ ]:
for label, (_, suffix) in VARIANTS.items():
    for what, action in [
        ('model', lambda s=suffix: gds.model.get(f'oulad-model-{s}').drop()),
        ('pipeline', lambda s=suffix: gds.pipeline.node_classification.get(
            f'oulad-pipeline-{s}').drop()),
    ]:
        try:
            action()
            print(f'{what} for {label!r} dropped')
        except Exception as error:
            print(f'{what} for {label!r}: {str(error)[:90]}')
try:
    G.drop()
    print('projection dropped')
except Exception as error:
    print('projection:', error)

gds.delete()
print('session deleted')

with driver.session(database=NEO4J_DATABASE) as session:
    totals = session.run('MATCH (n) WITH count(n) AS nodes '
                         'MATCH ()-[r]->() RETURN nodes, count(r) AS relationships').single()
driver.close()
print(f"\ngraph totals: {totals['nodes']:,} nodes, "
      f"{totals['relationships']:,} relationships (unchanged)")
print('sessions remaining:', [s.name for s in sessions.list()] or 'none')

## What this shows, and what it does not

**What it shows.** Whether a student's position in the engagement-and-demographics graph — with
no assessment data — carries enough signal to anticipate the module outcome, and whether a
128-dimensional embedding earns its cost against one `sum()`.

**What it does not.** Causation, obviously. Also not much about *why*: a FastRP vector is a
random projection, so individual dimensions mean nothing and the model cannot be read as an
explanation. If you need to tell a tutor why a student was flagged, train on interpretable
features — click volume, active days, last-active day — and use this only as a check on
whether the graph adds anything beyond them.

**Where the leakage line sits.** Assessment relationships were left out because scores
determine the outcome. Two other things sit closer to the line than they look:
`date_unregistration` on `CONTAINS_COURSE` is effectively the withdrawal label, and clicks
recorded *after* a student stopped participating leak the same information backwards. A
genuine early-warning model would cut every feature at a fixed day — say day 30 — and predict
from that alone. That is the natural next step, and it is a `WHERE r.date <= 30` in the
projection query.